# NFC tags — NDEF records and Type 2 tag memory

A virtual PC/SC reader with a simulated NTAG213 tag. With a real contactless reader (ACR122U, ACR1252U…), replace the virtual reader with `PcscNfcReader.Open()`; the rest of the code stays the same.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.21.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.21.0-preview.1"

## NDEF records
A URI uses a one-byte prefix code (0x04 = https://), a Text record carries its language, and a Smart Poster nests both.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Nfc;

var ndef = new NdefMessage([NdefRecord.SmartPoster("https://docs.example.com/p7", "Pump P-0007 manual"), NdefRecord.Text("Asset P-0007")]);
var ndefBytes = ndef.Encode();
Console.WriteLine(Convert.ToHexString(ndefBytes));
foreach (var f in NdefMessage.Describe(ndefBytes)) Console.WriteLine($"{f.Name,-16} {f.Value}");
var wifi = NdefRecord.WifiCredential(new WifiCredential("Plant-Commissioning", "example-only-key"));
Console.WriteLine(wifi);   // the key is never shown

## Read a tag
GET DATA returns the UID, READ BINARY four pages at a time; the data area starts at page 4 with TLVs.

In [ ]:
var nfcReader = new VirtualNfcReader();
var pumpTag = new VirtualType2Tag(Type2TagKind.Ntag213, [0x04, 0x51, 0x7A, 0x22, 0x9C, 0x61, 0x80], ndef);
nfcReader.Present(pumpTag);
await using var card = await nfcReader.WaitForTagAsync();
var tagClient = new Type2TagClient(card);
Console.WriteLine($"UID {Convert.ToHexString(await tagClient.GetUidAsync())}");
var (cc, dataArea) = await tagClient.ReadDataAreaAsync();
Console.WriteLine($"CC {Convert.ToHexString(cc)}: {cc[2] * 8} bytes");
Console.WriteLine(await tagClient.ReadNdefAsync());

## The memory, page by page

In [ ]:
var dump = await tagClient.DumpAsync();
var regions = Type2Tag.Map(dump, 144);
for (var page = 0; page < 12; page++) Console.WriteLine($"{page,2}  {Convert.ToHexString(dump, page * 4, 4)}  {regions[page * 4]}");

## Writing is opt-in
Writes are refused until allowed; only changed pages are written, and the NDEF TLV header is written last.

In [ ]:
try { await tagClient.WriteNdefAsync(new NdefMessage([NdefRecord.Text("x")])); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
var tagWriter = new Type2TagClient(card, new NfcTagOptions().AllowWrites());
var before = pumpTag.PagesWritten;
await tagWriter.WriteNdefAsync(new NdefMessage([.. ndef.Records, NdefRecord.Text("Serviced 2026-10-09 · tech 14")]));
Console.WriteLine($"{pumpTag.PagesWritten - before} pages written");
Console.WriteLine(await tagClient.ReadNdefAsync());

## Going further
`iotcom nfc readers`, `iotcom nfc read --sim --dump`, `iotcom nfc write --uri https://example.com --sim --allow-write`, `iotcom nfc decode <hex>`, the Gallery's *NFC asset tags* and the NfcTagReader sample. See `docs/en/protocols/nfc.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*